In [1]:
# =============================================================================
# CELL 1: CONFIGURATION
# =============================================================================

granularity = 'm'

START_DATE = '2025-10-07'
END_DATE = None

run_every_query = True

DATE_COL_MAP = {'q': 'book_date', 'm': 'book_date', 'w': 'app_date'}

LOBS = ['STE']

BASELINES = {
    'STE': {'ltv': 1.94, 'new_recovery_unadjusted': 0.557, 'apr': 0.25},
}

MODEL_PARAMS = {
    'mmi_standard_increase': 1.03,
    'expected_years_on_book': 2,
    'impound_probability': 0.15,
    'mean_unit_loss': 0.5,
    'unit_loss_to_model_score': 0.02,
    'skip_rate': 0.75,
    'kmx_loss_scale': 0.067,
}

PRICING_SCALAR = 1.0
USE_STE_METRICS = True
EXCLUDED_VINTAGES = {}

HURDLES = ['1) Lower', '2) Higher']

In [2]:
# =============================================================================
# CELL 2: IMPORTS AND DERIVED CONFIGURATION
# =============================================================================
import pandas as pd
import numpy as np
import pyodbc
import pickle
import warnings
import time
import datetime as dt
import re
import os
import openpyxl
from tqdm.notebook import tqdm

tqdm.pandas()
pd.set_option('display.max_columns', 100)
pd.set_option('display.min_rows', 100)

date_col = DATE_COL_MAP[granularity]
start_date = pd.Timestamp(START_DATE)
end_date = pd.Timestamp(END_DATE) if END_DATE else pd.Timestamp.today().normalize()

PERIOD_FREQ_MAP = {'q': 'Q', 'm': 'M', 'w': 'W-SAT'}
period_freq = PERIOD_FREQ_MAP[granularity]

start_period = start_date.to_period(period_freq)
end_period = end_date.to_period(period_freq)

min_date_sql = f"'{START_DATE}'"

print(f"Granularity: {granularity}")
print(f"Date column: {date_col}")
print(f"Period range: {start_period} to {end_period}")
print(f"SQL min_date: {min_date_sql}")

Granularity: m
Date column: book_date
Period range: 2025-10 to 2026-06
SQL min_date: '2025-10-07'


In [3]:
# =============================================================================
# CELL 3: UTILITY FUNCTIONS
# =============================================================================

def run_sql(filename, sub_list=None, connection=None, filename_is_query=False):
    if sub_list is None:
        sub_list = []
    if filename_is_query:
        query = filename
    else:
        with open(filename, 'r') as file:
            query = file.read()
    for text, var in sub_list:
        query = query.replace(text, var)
    if connection is None:
        with pyodbc.connect("DSN=Redshift_prod_new") as conn:
            warnings.filterwarnings("ignore", category=UserWarning)
            df = pd.read_sql_query(sql=query, con=conn)
            warnings.filterwarnings("default", category=UserWarning)
            return df
    else:
        warnings.filterwarnings("ignore", category=UserWarning)
        df = pd.read_sql_query(sql=query, con=connection)
        warnings.filterwarnings("default", category=UserWarning)
        return df


def store_pickle(data, filename):
    if isinstance(data, str):
        data, filename = filename, data
    with open(filename, 'wb') as file:
        pickle.dump(data, file)


def get_pickle(filename):
    with open(filename, 'rb') as file:
        return pickle.load(file)


def cached_sql(filename, pickle_name, sub_list=None, connection=None, force_refresh=False):
    if force_refresh or not os.path.exists(pickle_name):
        df = run_sql(filename, sub_list=sub_list, connection=connection)
        if df.empty:
            print(f"  WARNING: query '{filename}' returned 0 rows")
        store_pickle(df, pickle_name)
        return df
    df = get_pickle(pickle_name)
    if df.empty:
        print(f"  WARNING: cached '{pickle_name}' contains 0 rows")
    return df


def weighted_average_and_sum(group, metrics):
    if isinstance(metrics, str):
        weighted_avg = (group[metrics] * group.amt_financed).sum() / group.amt_financed.sum()
        return pd.Series({metrics: weighted_avg, 'amt_financed': group.amt_financed.sum()})
    result_dict = {'amt_financed': group.amt_financed.sum()}
    for metric in metrics:
        weighted_avg = (group[metric] * group.amt_financed).sum() / group.amt_financed.sum()
        result_dict[metric] = weighted_avg
    return pd.Series(result_dict)


def format_vintage(period_series):
    if period_series.empty:
        return period_series.astype(str)
    freq = period_series.iloc[0].freqstr
    if freq == 'Q-DEC':
        return period_series.dt.year.astype(str) + ' Q' + period_series.dt.quarter.astype(str)
    elif freq == 'M':
        return period_series.dt.year.astype(str) + ' M' + period_series.dt.month.astype(str).str.zfill(2)
    return period_series.astype(str)


def _ste_weighted_avg(g, metric_col, weight_col='con_amount_financed_back'):
    mask = g[metric_col].notna()
    if not mask.any():
        return np.nan
    return (g.loc[mask, metric_col] * g.loc[mask, weight_col]).sum() / g.loc[mask, weight_col].sum()

In [4]:
# =============================================================================
# CELL 4: ULA MULTIPLIER FUNCTIONS
# =============================================================================

def get_ula_multiplier_nonkmx(ula_df, leave_out='None'):
    ula_df['loss_multiplier'] = 1.0

    if leave_out != 'Previous ACA chargeoff':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.prev_co_flag

    if leave_out != 'Small amount financed':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.small_amt_financed_flag * (1 - ula_df.pricing_change_flag)

    if leave_out != 'Zero cash down':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.zero_cash_down_flag * (1 - ula_df.pricing_change_flag)

    if leave_out != 'High mileage vehicle':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_mileage_vehicle_flag

    if leave_out != 'High PTI':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_pti_flag * (1 - ula_df.pricing_change_flag)

    if leave_out != 'Car make':
        ula_df.loss_multiplier *= (1 + 0.1 * ula_df.car_make_penalty_flag
                                   - 0.1 * ula_df.car_make_benefit_flag
                                   - 0.1 * ula_df.pricing_change_flag * ula_df.car_make_benefit_flag)

    if leave_out != 'Theft risk':
        ula_df.loss_multiplier *= (0.987 + 0.099 * ula_df.theft_risk_flag * (1 - ula_df.pricing_change_flag)
                                   + 0.013 * ula_df.pricing_change_flag)

    if leave_out != 'MCY high model score, low mileage':
        ula_df.loss_multiplier *= 1 - 0.2 * ula_df.mcy_low_mileage_flag

    if leave_out != 'Weekday/weekend decision':
        ula_df.loss_multiplier *= 1 - 0.05 * ula_df.weekend_flag + 0.02 * ula_df.weekday_flag

    if leave_out != 'Student Loans':
        ula_df.loss_multiplier *= 1 + (0.1 * ula_df.student_loan_flag
            - np.minimum(np.maximum((ula_df.cd_model_score - (130 - 2 * ula_df.ent_flag)) * 0.006, 0), 0.03)
            * (1 - ula_df.student_loan_flag)) * ula_df.student_loans_cutoff_date

    if leave_out != 'Low PTI':
        ula_df.loss_multiplier *= 1 + (-0.03 * np.minimum(ula_df.cd_model_score, 135) + 3.9) * ula_df.low_pti_flag

    if leave_out != 'Secured credit (Chime, etc.)':
        ula_df.loss_multiplier *= 0.966 + 0.273 * ula_df.nonkmx_chime_flag

    if leave_out != 'Employment type':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.seasonal_employment_flag - 0.1 * ula_df.waiter_employment_flag

    if leave_out != 'Authorized tradelines':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.nonkmx_auth_tradelines_flag

    if leave_out != 'Fraud':
        ula_df.loss_multiplier *= 1 + (ula_df.fraud_adjustment - 1)

    if leave_out != 'Driver flag':
        ula_df.loss_multiplier *= 1 + 0.15 * ula_df.driver_flag

    if leave_out != 'Clip':
        ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.8, 2)

    if leave_out != 'Dealer Level (Non-KMX)':
        ula_df.loss_multiplier *= ula_df.pricing_scalar
        ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.7, 1.4)

    return ula_df

In [5]:
# =============================================================================
# CELL 5: DATA FETCH (SQL + PICKLE + MAXLTV)
# =============================================================================

os.makedirs('cache', exist_ok=True)
force = run_every_query

need_conn = force or not all(
    os.path.exists(p) for p in ('cache/ste_ula_v1.pkl', 'cache/ste_recovery_v1.pkl', 'cache/ste_weekly_v1.pkl')
)

if need_conn:
    conn = pyodbc.connect("DSN=Redshift_prod_new")

    with open('ste_ragu_temptables.txt', 'r') as f:
        conn.execute(f.read().strip())
    print('Temp tables created')

    ula_df_total = cached_sql('ste_ragu_ula.txt', 'cache/ste_ula_v1.pkl',
                              connection=conn, force_refresh=force)
    print(f'ULA ready: {len(ula_df_total):,} records')

    new_recovery = cached_sql('ste_ragu_recovery.txt', 'cache/ste_recovery_v1.pkl',
                              connection=conn, force_refresh=force)
    print(f'Recovery ready: {len(new_recovery):,} records')

    ste_weekly_raw = cached_sql('ste_ragu_weekly.txt', 'cache/ste_weekly_v1.pkl',
                                connection=conn, force_refresh=force)
    print(f'STE weekly metrics ready: {len(ste_weekly_raw):,} records')

    # Fetch maxltv for hurdle assignment
    lrn_query = """
    SELECT ldcf.account_number, lrn.maxltv
    FROM los_deal_current_fact ldcf
    LEFT JOIN sandbox.loan_random_numbers lrn
        ON lrn.loan_id = ldcf.loan_id
    WHERE ldcf.data_source_id = 101
      AND ldcf.application_received_dtm >= '2025-10-07'
    """
    lrn_df = pd.read_sql_query(lrn_query, conn)
    print(f'LRN (maxltv) ready: {len(lrn_df):,} records')

    conn.close()
else:
    ula_df_total = get_pickle('cache/ste_ula_v1.pkl')
    new_recovery = get_pickle('cache/ste_recovery_v1.pkl')
    ste_weekly_raw = get_pickle('cache/ste_weekly_v1.pkl')
    lrn_df = run_sql(lrn_query)
    print('ULA/Recovery/Weekly loaded from cache; LRN queried fresh')

print(f"ULA records: {len(ula_df_total):,}")
print(f"LRN records: {len(lrn_df):,}")

Temp tables created
ULA ready: 68,671 records
Recovery ready: 15,997 records
STE weekly metrics ready: 16,478 records


C:\Users\ahmed.ali\AppData\Local\Temp\ipykernel_31080\670326392.py:40: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  lrn_df = pd.read_sql_query(lrn_query, conn)


LRN (maxltv) ready: 550,207 records
ULA records: 68,671
LRN records: 550,207


In [6]:
# =============================================================================
# CELL 6: HURDLE ASSIGNMENT + PERIOD FILTERING
# =============================================================================

# Filter out Core LOB if present
ula_df_total = ula_df_total[ula_df_total.lob != 'Core']
ula_df_total['lob'] = 'STE'

# Merge maxltv and assign hurdle
ula_df_total = ula_df_total.merge(
    lrn_df[['account_number', 'maxltv']].drop_duplicates('account_number'),
    on='account_number', how='left'
)
ula_df_total['hurdle'] = np.where(
    ula_df_total['maxltv'].fillna(0) < 500,
    '2) Higher', '1) Lower'
)
print(f"Hurdle distribution:\n{ula_df_total['hurdle'].value_counts()}")
print(f"maxltv coverage: {ula_df_total['maxltv'].notna().sum()} / {len(ula_df_total)} ({ula_df_total['maxltv'].notna().mean():.1%})")

# Date type conversion
ula_df_total['app_date'] = pd.to_datetime(ula_df_total['app_date'])
ula_df_total['book_date'] = pd.to_datetime(ula_df_total['book_date'])
new_recovery['app_date'] = pd.to_datetime(new_recovery['app_date'])
new_recovery['book_date'] = pd.to_datetime(new_recovery['book_date'])

# Assign period columns
for df in [ula_df_total, new_recovery]:
    df['quarter'] = pd.to_datetime(df[date_col]).dt.to_period('Q')
    df['month'] = pd.to_datetime(df[date_col]).dt.to_period('M')
    df['week'] = pd.to_datetime(df[date_col]).dt.to_period('W-SAT')

    period_key = {'q': 'quarter', 'm': 'month', 'w': 'week'}
    df['period'] = df[period_key[granularity]]

# Filter to configured date range
for df in [ula_df_total, new_recovery]:
    mask = (df['period'] >= start_period) & (df['period'] <= end_period)
    df.drop(df[~mask].index, inplace=True)

ula_df_total['book_week'] = ula_df_total['book_week'].astype(str)
ula_df_total[f'{date_col}_str'] = ula_df_total[date_col].astype(str)

# STE-specific caps
ula_df_total['bbltv'] = ula_df_total.amt_financed / ula_df_total.bbvalue.replace(0, np.nan)
ula_df_total = ula_df_total[
    (ula_df_total.bbltv <= 10.0) |
    (ula_df_total.bbvalue.isna()) |
    (ula_df_total.bbvalue == 0)
]
ula_df_total = ula_df_total[ula_df_total.pti <= 0.6]
ula_df_total = ula_df_total[ula_df_total.total_income <= 200000]

print(f"\nPeriods in data: {ula_df_total['period'].nunique()}")
print(f"Period range: {ula_df_total['period'].min()} to {ula_df_total['period'].max()}")
print(f"ULA after caps: {len(ula_df_total):,}")
print(f"\nHurdle x Period counts:")
print(ula_df_total.groupby(['hurdle', 'period']).size().unstack(fill_value=0))

Hurdle distribution:
hurdle
1) Lower     35645
2) Higher    33026
Name: count, dtype: int64
maxltv coverage: 68152 / 68671 (99.2%)

Periods in data: 8
Period range: 2025-10 to 2026-05
ULA after caps: 67,379

Hurdle x Period counts:
period     2025-10  2025-11  2025-12  2026-01  2026-02  2026-03  2026-04  \
hurdle                                                                     
1) Lower      4198     6869     7369     2588     2362     5637     3661   
2) Higher     4689     6082     5838     2524     2128     4853     3527   

period     2026-05  
hurdle              
1) Lower      2492  
2) Higher     2562  


In [7]:
# =============================================================================
# CELL 7: FLAG CREATION, DATA REFINEMENT, ms_df, DUAL-PATH METRICS
# =============================================================================

date_col_str = f'{date_col}_str'

# --- ULA Processing ---
ula_df_total['mtn_3_1_flag'] = ula_df_total.mtn_model == 'MTN3.1'
ula_df_total['vehicle_age'] = np.maximum(ula_df_total[date_col_str].str[:4].astype(int) - ula_df_total.model_year, 1/365)
ula_df_total.tradein_value = ula_df_total.tradein_value.fillna(0)
ula_df_total.make = ula_df_total.make.str.upper().str[:3]
ula_df_total.lob_or_bucket = np.select(
    [ula_df_total.lob_or_bucket.isna() & ula_df_total.lob.isin(['Core', 'FRN']),
     ula_df_total.lob_or_bucket.isna() & ~ula_df_total.lob.isin(['Core', 'FRN'])],
    ['D', 'C'], default=ula_df_total.lob_or_bucket)
ula_df_total['continuous_vehicle_age'] = (ula_df_total[date_col_str].str[:4].astype(int)
    + ula_df_total[date_col_str].str[5:7].astype(int) / 12
    - (ula_df_total.model_year - 0.25) - 1)

ula_df_total['pricing_scalar'] = PRICING_SCALAR

# --- Driver Flag ---
warnings.filterwarnings("ignore", category=UserWarning)
ula_df_total.job_company = ula_df_total.job_company.fillna('')
ula_df_total['driver_flag'] = np.where(
    ula_df_total.job_company.str.contains('(LYFT)|(UBER)|(GRUB ?HUB)|(DOOR ?DASH)|(GO ?PUFF)|(POST ?MATE)|(INSTA ?CART)|(DOMINO)|(PAPA J)|(PIZZA)|(JIMMY ?JOHN)|(SELF)'),
    1, 0)
warnings.filterwarnings("default", category=UserWarning)

# --- NA Handling ---
ula_df_total = ula_df_total.dropna(subset=['lob'])
ula_df_total.cash_down = ula_df_total.cash_down.fillna(0)
ula_df_total.specialty_dealer = ula_df_total.specialty_dealer.fillna('not specialty')
ula_df_total.prev_co_count = ula_df_total.prev_co_count.fillna(0)

# --- NonKMX Flags ---
ula_df_total['ent_fld_flag'] = False
ula_df_total['small_amt_financed_flag'] = (ula_df_total.bbvalue > 0) & (ula_df_total.bbvalue < 5000) & (ula_df_total.amt_financed < 4500)
ula_df_total['zero_cash_down_flag'] = (ula_df_total.cash_down <= 250) & (ula_df_total.tradein_value < 3000)
ula_df_total['high_mileage_vehicle_flag'] = (ula_df_total.mileage >= 100000) & (ula_df_total.cd_model_score < 130)
ula_df_total['high_pti_flag'] = (ula_df_total.pti > 0.2) & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['car_make_penalty_flag'] = ula_df_total.make.isin({'CAD', 'CHR', 'BMW', 'BUI', 'SUB'}) & (ula_df_total.cd_model_score < 130) & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['car_make_benefit_flag'] = (ula_df_total.cd_model_score >= 133) & ula_df_total.make.isin({'HON', 'TOY', 'LEX'}) & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['theft_risk_flag'] = ula_df_total.make.isin({'KIA', 'HYU'}) & (ula_df_total.specialty_dealer != 'Ally') & (ula_df_total.model_year >= 2015) & (ula_df_total.model_year <= 2021) & (ula_df_total[date_col_str] >= '2022-07-01') & (ula_df_total[date_col_str] < '2025-01-01')
ula_df_total['mcy_low_mileage_flag'] = False
ula_df_total['weekend_flag'] = ula_df_total.day_of_week.isin([0, 6])
ula_df_total['weekday_flag'] = ula_df_total.day_of_week.isin(range(1, 6))
ula_df_total['secured_credit_flag'] = ula_df_total.secured_credit_card
ula_df_total['chime_flag'] = ula_df_total.chime_indicator
ula_df_total['nonkmx_chime_flag'] = ula_df_total.nonkmx_chime_indicator
ula_df_total['seasonal_employment_flag'] = ula_df_total['seasonal_employment_flag'] == 1 if 'seasonal_employment_flag' in ula_df_total.columns else (ula_df_total.get('employment', '') == 'seasonal')
ula_df_total['waiter_employment_flag'] = ula_df_total['waiter_employment_flag'] == 1 if 'waiter_employment_flag' in ula_df_total.columns else (ula_df_total.get('employment', '') == 'waiter')
ula_df_total['nonkmx_auth_tradelines_flag'] = ula_df_total.pct_auth_tradelines > 0.2
ula_df_total['prev_co_flag'] = ula_df_total.prev_co_count > 0
ula_df_total['null_fico_w_vantage_flag'] = ((ula_df_total.fico_score < 300) | (ula_df_total.fico_score > 850)) & (ula_df_total.vantage_score >= 300) & (ula_df_total.vantage_score <= 850)
ula_df_total['pricing_change_flag'] = ula_df_total[date_col_str] >= '2024-10-01'
ula_df_total['ent_flag'] = False
ula_df_total['student_loans_cutoff_date'] = ula_df_total[date_col_str] >= '2023-05-01'
ula_df_total['low_pti_flag'] = (ula_df_total.pti <= 0.05) & (ula_df_total.cd_model_score >= 130) & (ula_df_total.cb_flag)
ula_df_total['student_loan_flag'] = 0

# --- Deduplicate driver flags ---
combined_driver_flag_df = ula_df_total.groupby('account_number').driver_flag.max().reset_index()
ula_df_total = ula_df_total.drop(columns='driver_flag').merge(combined_driver_flag_df, on='account_number')
ula_df_total = ula_df_total.drop(columns='job_company').drop_duplicates()

# --- Vintage strings ---
ula_df_total['vintage'] = format_vintage(ula_df_total['period'])
new_recovery['vintage'] = format_vintage(new_recovery['period'])

# --- Model scores aggregated per hurdle ---
ms_df = ula_df_total.groupby(['period', 'lob', 'hurdle']).apply(
    weighted_average_and_sum, 'cd_model_score', include_groups=False
).reset_index()
ms_df = ms_df.rename(columns={'cd_model_score': 'model_score'})
ms_df['period'] = format_vintage(ms_df['period'])

print(f"ULA after refinement: {len(ula_df_total):,}")
print(f"Model scores aggregated: {len(ms_df)} period-LOB-hurdle combinations")

# --- DUAL-PATH: Build ste_metrics_df per hurdle ---
ste_filtered = ste_weekly_raw.copy()
ste_date_col = 'application_received_dtm' if granularity == 'w' else 'book_date'
ste_filtered[ste_date_col] = pd.to_datetime(ste_filtered[ste_date_col])
ste_filtered['period'] = ste_filtered[ste_date_col].dt.to_period(period_freq)
ste_filtered = ste_filtered[(ste_filtered.period >= start_period) & (ste_filtered.period <= end_period)]
ste_filtered['vintage'] = format_vintage(ste_filtered['period'])

# Apply same STE caps
ste_filtered = ste_filtered[ste_filtered['con_pti_back'] <= 0.6]
ste_filtered = ste_filtered[ste_filtered['total_income'] <= 200000]
ste_filtered['bbltv'] = ste_filtered['con_amount_financed_back'] / ste_filtered['bb_value'].replace(0, np.nan)
ste_filtered = ste_filtered[
    (ste_filtered['bbltv'] <= 10.0) |
    (ste_filtered['bb_value'].isna()) |
    (ste_filtered['bb_value'] == 0)
]

# Merge hurdle onto ste_filtered via lrn_df
ste_filtered = ste_filtered.merge(
    lrn_df[['account_number', 'maxltv']].drop_duplicates('account_number'),
    on='account_number', how='left'
)
ste_filtered['hurdle'] = np.where(
    ste_filtered['maxltv'].fillna(0) < 500,
    '2) Higher', '1) Lower'
)

def _build_ste_metrics(g):
    w = g['con_amount_financed_back']
    ms_valid = g['con_risk_model_score'].notnull()
    ltv_valid = g['bbltv'].notnull()
    return pd.Series({
        'model_score_wtd': (g.loc[ms_valid, 'con_risk_model_score'] * w[ms_valid]).sum() / w[ms_valid].sum() if ms_valid.any() else np.nan,
        'ltv_wtd': (g.loc[ltv_valid, 'bbltv'] * w[ltv_valid]).sum() / w[ltv_valid].sum() if ltv_valid.any() else np.nan,
        'apr_wtd': (g['con_apr'] * w).sum() / w.sum(),
        'amt_financed_total': w.sum(),
        'n_accounts': len(g),
    })

ste_metrics_df = ste_filtered.groupby(['vintage', 'hurdle']).apply(_build_ste_metrics).reset_index()
print(f"\nste_metrics_df built: {len(ste_metrics_df)} vintage-hurdle combinations")
print(ste_metrics_df)

ULA after refinement: 16,030
Model scores aggregated: 16 period-LOB-hurdle combinations

ste_metrics_df built: 16 vintage-hurdle combinations
     vintage     hurdle  model_score_wtd   ltv_wtd   apr_wtd  \
0   2025 M10   1) Lower       131.037004  1.513452  0.235452   
1   2025 M10  2) Higher       131.005325  1.532219  0.237569   
2   2025 M11   1) Lower       131.444309  1.550942  0.233455   
3   2025 M11  2) Higher       131.272689  1.537159  0.234775   
4   2025 M12   1) Lower       131.498208  1.577746  0.233681   
5   2025 M12  2) Higher       131.828834  1.568506  0.235247   
6   2026 M01   1) Lower       132.164958  1.560799  0.234543   
7   2026 M01  2) Higher       132.344720  1.514535  0.234365   
8   2026 M02   1) Lower       132.860784  1.513869  0.233536   
9   2026 M02  2) Higher       133.384205  1.486220  0.232861   
10  2026 M03   1) Lower       134.172425  1.471018  0.227632   
11  2026 M03  2) Higher       134.633297  1.450986  0.229452   
12  2026 M04   1) Lower   

In [8]:
# =============================================================================
# CELL 8: RAGU SCORE COMPUTATION (PARTITIONED BY HURDLE x MONTH)
# =============================================================================

mean_unit_loss = MODEL_PARAMS['mean_unit_loss']
unit_loss_to_model_score = MODEL_PARAMS['unit_loss_to_model_score']


def get_ragu_score(vintage, lob, hurdle, ula_df_total, new_recovery, ms_df, baseline_config, ste_metrics_df=None):
    """RAGU Score for a single vintage, LOB, and hurdle pool."""
    baseline_ltv = baseline_config['ltv']
    new_baseline_recovery_unadjusted_pct = baseline_config['new_recovery_unadjusted']
    baseline_apr = baseline_config['apr']

    ula_df = ula_df_total[
        (ula_df_total.vintage == vintage) &
        (ula_df_total.lob == lob) &
        (ula_df_total.hurdle == hurdle)
    ].copy()

    if len(ula_df) == 0:
        return None

    ula_df = get_ula_multiplier_nonkmx(ula_df)

    ula_df = ula_df[['account_number', date_col, 'bbvalue', 'sale_price', 'amt_financed', 'lob_or_bucket', 'lob', 'hurdle', 'loss_multiplier', 'apr']]

    nr = new_recovery[['account_number', 'new_recovery_multiplier']].drop_duplicates(subset='account_number', keep='first')
    mix_df = ula_df.merge(nr.rename(columns={'new_recovery_multiplier': 'recovery_multiplier'}),
                          on='account_number', how='left').drop_duplicates(subset='account_number', keep='first')

    mix_df['ltv'] = mix_df.amt_financed / mix_df.bbvalue

    bb_populated_df = mix_df[mix_df['bbvalue'].notna() & (mix_df['bbvalue'] > 0)]
    if len(bb_populated_df) == 0:
        return None

    full_pop_metrics = bb_populated_df.groupby('lob').apply(
        weighted_average_and_sum,
        ['loss_multiplier', 'ltv', 'bbvalue'],
        include_groups=False
    )

    apr_metrics = mix_df.groupby('lob').apply(
        weighted_average_and_sum, 'apr', include_groups=False
    ).drop(columns='amt_financed')

    recovery_df = mix_df[mix_df['bbvalue'].notna() & (mix_df['bbvalue'] > 0) & mix_df['recovery_multiplier'].notna()]
    recovery_df = recovery_df.copy()
    recovery_df['recovery_unadjusted_multiplier'] = recovery_df['recovery_multiplier']
    recovery_metrics = recovery_df.groupby('lob').apply(
        weighted_average_and_sum,
        ['recovery_unadjusted_multiplier'],
        include_groups=False
    )
    recovery_metrics = recovery_metrics.drop(columns='amt_financed')

    grouped_mix_df = full_pop_metrics.join(recovery_metrics).join(apr_metrics)

    vintage_ms_df = ms_df.loc[
        (ms_df['period'] == vintage) & (ms_df['hurdle'] == hurdle),
        ['lob', 'model_score']
    ].copy()

    index_name = grouped_mix_df.index.name
    if isinstance(index_name, str) and index_name in grouped_mix_df.columns:
        grouped_mix_df = grouped_mix_df.reset_index(drop=True)
    else:
        grouped_mix_df = grouped_mix_df.reset_index()

    full_df = grouped_mix_df.merge(vintage_ms_df, on='lob')

    if len(full_df) == 0:
        return None

    full_df['est_unit_loss'] = mean_unit_loss
    full_df['unit_loss_score'] = full_df.model_score + (1 - full_df.loss_multiplier) * full_df.est_unit_loss / unit_loss_to_model_score
    full_df = full_df.set_index('lob')

    full_df['ms_original'] = full_df.model_score.copy()
    full_df['baselined_unadjusted_recovery'] = (full_df.recovery_unadjusted_multiplier / new_baseline_recovery_unadjusted_pct).copy()

    full_df['gross_loss_impact'] = full_df['unit_loss_score'] - full_df['ms_original']
    full_df['recovery_impact'] = (full_df['unit_loss_score'] * full_df['est_unit_loss']
        * full_df['recovery_unadjusted_multiplier'] * (full_df['baselined_unadjusted_recovery'] - 1))
    full_df['ltv_impact'] = ((baseline_ltv / full_df['ltv']) - 1) * 17
    full_df['apr_impact'] = (baseline_apr - full_df['apr']) / 0.01 * 0.7
    full_df['ragu_score'] = (
        (1 - full_df.est_unit_loss * full_df.recovery_unadjusted_multiplier) * full_df.unit_loss_score
        + full_df.est_unit_loss * full_df.recovery_unadjusted_multiplier
        * full_df.unit_loss_score * full_df.baselined_unadjusted_recovery
        + full_df['ltv_impact']
        + full_df['apr_impact'])

    # STE dual-path override
    if ste_metrics_df is not None:
        ste_row = ste_metrics_df[(ste_metrics_df.vintage == vintage) & (ste_metrics_df.hurdle == hurdle)]
        if len(ste_row) > 0:
            ste_row = ste_row.iloc[0]
            full_df['ms_original'] = ste_row['model_score_wtd']
            full_df['ltv'] = ste_row['ltv_wtd']
            full_df['apr'] = ste_row['apr_wtd']
            full_df['ltv_impact'] = ((baseline_ltv / full_df['ltv']) - 1) * 17
            full_df['apr_impact'] = (baseline_apr - full_df['apr']) / 0.01 * 0.7
            full_df['ragu_score'] = (
                (1 - full_df.est_unit_loss * full_df.recovery_unadjusted_multiplier) * full_df.unit_loss_score
                + full_df.est_unit_loss * full_df.recovery_unadjusted_multiplier
                * full_df.unit_loss_score * full_df.baselined_unadjusted_recovery
                + full_df['ltv_impact']
                + full_df['apr_impact'])

    full_df['vintage'] = vintage
    full_df['hurdle'] = hurdle
    return full_df


# --- Main RAGU Loop: STE x Periods x Hurdles ---
all_vintages = sorted(ula_df_total['vintage'].unique())
results = []

for lob in LOBS:
    excluded = EXCLUDED_VINTAGES.get(lob, set())
    baseline_config = BASELINES[lob]

    for vintage in all_vintages:
        if vintage in excluded:
            continue
        for hurdle in HURDLES:
            try:
                result = get_ragu_score(
                    vintage, lob, hurdle, ula_df_total, new_recovery, ms_df, baseline_config,
                    ste_metrics_df=ste_metrics_df
                )
                if result is not None:
                    results.append(result)
            except Exception as e:
                print(f"Error: {vintage} {lob} {hurdle}: {e}")

    print(f"{lob} complete")

if results:
    all_df = pd.concat(results, ignore_index=False)
    all_df = all_df.reset_index()
    print(f"\nTotal results: {len(all_df)} rows across {all_df.vintage.nunique()} vintages x {all_df.hurdle.nunique()} hurdles")
else:
    print("WARNING: No results produced")
    all_df = pd.DataFrame()

print("[PROGRESS] Scoring Complete")

STE complete

Total results: 16 rows across 8 vintages x 2 hurdles
[PROGRESS] Scoring Complete


In [9]:
# =============================================================================
# CELL 9: EXCEL EXPORT (ste_hurdle_ragu.xlsx)
# =============================================================================

METRIC_ROWS = [
    ('Model Score',                'ms_original'),
    ('Expected Gross Loss Impact', 'gross_loss_impact'),
    ('Recovery Impact',            'recovery_impact'),
    ('LTV Impact',                 'ltv_impact'),
    ('APR Impact',                 'apr_impact'),
    ('RAGU Score',                 'ragu_score'),
    ('Amount Financed',            'amt_financed'),
    ('Weighted LTV',               'ltv'),
    ('Weighted APR',               'apr'),
]

EXCEL_SHEET_MAP = {'q': 'Data Tables (Q)', 'm': 'Data Tables (M)', 'w': 'Data Tables (W)'}
EXCEL_OUTPUT = 'ste_hurdle_ragu.xlsx'

sheet_name = EXCEL_SHEET_MAP[granularity]
sorted_vintages = sorted(all_df['vintage'].unique())

if os.path.exists(EXCEL_OUTPUT):
    wb = openpyxl.load_workbook(EXCEL_OUTPUT)
    if sheet_name in wb.sheetnames:
        del wb[sheet_name]
    ws = wb.create_sheet(sheet_name)
else:
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = sheet_name

current_row = 1

for hurdle in HURDLES:
    hurdle_data = all_df[all_df.hurdle == hurdle].set_index('vintage')

    ws.cell(row=current_row, column=1, value=f'STE - {hurdle}')
    for col_idx, v in enumerate(sorted_vintages, start=2):
        ws.cell(row=current_row, column=col_idx, value=v)
    current_row += 1

    for label, col_key in METRIC_ROWS:
        ws.cell(row=current_row, column=1, value=label)
        for col_idx, v in enumerate(sorted_vintages, start=2):
            if v in hurdle_data.index:
                val = hurdle_data.loc[v, col_key]
                if hasattr(val, 'iloc'):
                    val = val.iloc[0]
                ws.cell(row=current_row, column=col_idx, value=val)
        current_row += 1

    current_row += 1

wb.save(EXCEL_OUTPUT)
print(f"Saved to {EXCEL_OUTPUT} (sheet: {sheet_name})")
print(f"  {len(HURDLES)} hurdle groups x {len(sorted_vintages)} periods")
print("[PROGRESS] Excel Export Complete")

Saved to ste_hurdle_ragu.xlsx (sheet: Data Tables (M))
  2 hurdle groups x 8 periods
[PROGRESS] Excel Export Complete
